In [1]:
import torch
import ChemSurrogate.data_processing as dp
from ChemSurrogate.configs import (
    DatasetConfig,
    AEConfig,
    EMConfig,
    )
import ChemSurrogate.analysis as analysis
from ChemSurrogate.trainer import (
    load_autoencoder_objects,
    load_emulator_objects,
    load_skipcon_emulator_objects,
    )
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
import pandas as pd
import os
import gc
import re
import random
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def generate_stoichiometric_matrix():
    """
    Generates a stoichiometric matrix for the elements in the dataset.
    An unscaled vector of the species multiplied by this matrix will give the elemental abundances, which are conserved.
    Additionally tracks BULK and SURFACE stoichiometric.
    """
    elements = ["H", "HE", "C", "N", "O", "S", "SI", "MG", "CL"]
    stoichiometric_matrix = np.zeros((len(elements), DatasetConfig.num_species))
    modified_species = [s.replace("BULK_", "").replace("SURF_", "") for s in DatasetConfig.species]
    
    elements_patterns = {
        'H': re.compile(r'H(?!E)(\d*)'),
        'HE': re.compile(r'HE(\d*)'),
        'C': re.compile(r'C(?!L)(\d*)'),
        'N': re.compile(r'N(\d*)'),
        'O': re.compile(r'O(\d*)'),
        'S': re.compile(r'S(?!I)(\d*)'),
        'SI': re.compile(r'SI(\d*)'),
        'MG': re.compile(r'MG(\d*)'),
        'CL': re.compile(r'CL(\d*)'),
    }

    for element, pattern in elements_patterns.items():
        elem_index = elements.index(element)
        for i, species in enumerate(modified_species):
            match = pattern.search(species)
            if match and species not in ["SURFACE", "BULK"]:
                multiplier = int(match.group(1)) if match.group(1) else 1
                stoichiometric_matrix[elem_index, i] = multiplier
        
    return stoichiometric_matrix.T

In [ ]:
molecular_weights = [1.008, 4.0026, 12.011, 14.007, 15.999, 32.06, 28.085, 24.305, 35.45]

conservation_matrix = dp.generate_stoichiometric_matrix()
z = conservation_matrix @ molecular_weights

print(z.shape)

np.save("molecular_weights.npy", z)

(333,)
